# Neuroevolucion con el dataset de vocales por persona

Los cinco folds contienen solo personas de `train`. `val` permite seguir el rendimiento por generacion. El unico `test` global se carga despues de elegir la arquitectura.

En `neuroevolution/config.py`, `CONFIG["input_modality"]` elige `"audio"` (onda PCM, CNN 1D) o `"spectrogram"` (espectrograma log-mel, CNN 2D). `"image"` sigue aceptado como alias antiguo de `"spectrogram"`. Ejecuta primero `vowels_dataset_por_persona.ipynb`. Las clases son `0 = Control` y `1 = Patologicas`.

## 1. Preparación e importaciones

In [ ]:
# Install required packages if needed
import os
from neuroevolution import CONFIG, install_packages, setup_logging

MODALITY_NAME = 'spectrogram' if CONFIG['input_modality'] == 'image' else CONFIG['input_modality']
ARTIFACTS_DIR = os.path.join('artifacts', f'test_vowels_{MODALITY_NAME}')
setup_logging(os.path.join(ARTIFACTS_DIR, 'execution_log.txt'), echo_to_console=False)

packages = [
    "torch==2.11.0",
    "numpy>=1.21.0",
    "matplotlib>=3.5.0",
    "seaborn>=0.11.0",
    "scikit-learn>=1.0.0"
]

install_packages(packages)

In [ ]:
import csv
import json
import os
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

from neuroevolution import (
    CONFIG, setup_device, setup_seeds, setup_logging,
    HybridNeuroevolution, plot_fitness_evolution,
    show_evolution_statistics, analyze_failed_evaluations,
    configure_plot_style,
)
from neuroevolution.evaluation.cross_validation import evaluate_single_fold

print("Módulos importados")

from neuroevolution.config import configure_vowels_input


## 2. Configuración para los arrays del nuevo dataset

In [ ]:
def locate_dataset(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        path = candidate / "artifacts" / "vowels_dataset"
        if (path / "train.csv").is_file():
            return path
    raise FileNotFoundError("Ejecuta primero vowels_dataset_por_persona.ipynb")

DATASET_ROOT = locate_dataset(Path.cwd())
configure_vowels_input(CONFIG, str(DATASET_ROOT))
ARRAY_ROOT = DATASET_ROOT / CONFIG['fold_files_subdirectory']
CONFIG['artifacts_dir'] = ARTIFACTS_DIR

log_path = os.path.join(CONFIG['artifacts_dir'], 'execution_log.txt')
setup_logging(log_path, echo_to_console=False)
device = setup_device()
setup_seeds(42)
configure_plot_style()
print("Dataset:", DATASET_ROOT)
print("Modalidad:", CONFIG["input_modality"])
print("Arrays:", ARRAY_ROOT)
print("Dispositivo:", device)


## 3. Comprobar los cinco folds sin cargar test

La comprobación abre solo los arrays y CSV de `train` y validación interna. También prepara `val` para monitorizar cada generación. `X_test_vowels.npy` no se abre en esta sección.


In [ ]:
def csv_rows(name: str) -> list[dict]:
    with (DATASET_ROOT / name).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

global_train_rows = csv_rows("train.csv")
monitor_rows = csv_rows("val.csv")
train_people = {row['person_id'] for row in global_train_rows}
monitor_people = {row['person_id'] for row in monitor_rows}
assert len(train_people) == 70 and len(monitor_people) == 10
assert train_people.isdisjoint(monitor_people)

for fold in range(5):
    fold_train_rows = csv_rows(f"fold_{fold}_train.csv")
    fold_val_rows = csv_rows(f"fold_{fold}_val.csv")
    fold_train_people = {row['person_id'] for row in fold_train_rows}
    fold_val_people = {row['person_id'] for row in fold_val_rows}
    assert len(fold_train_people) == 56 and len(fold_val_people) == 14
    assert fold_train_people.isdisjoint(fold_val_people)
    assert fold_train_people | fold_val_people == train_people
    for rows, expected in ((fold_train_rows, 28), (fold_val_rows, 7)):
        counts = {
            label: len({row['person_id'] for row in rows if row['label'] == label})
            for label in ('Control', 'Patologicas')
        }
        assert counts == {'Control': expected, 'Patologicas': expected}
    suffix = f"_vowels_fold_{fold + 1}.npy"
    x_train = np.load(ARRAY_ROOT / f"X_train{suffix}", mmap_mode="r")
    y_train = np.load(ARRAY_ROOT / f"y_train{suffix}", mmap_mode="r")
    x_val = np.load(ARRAY_ROOT / f"X_val{suffix}", mmap_mode="r")
    y_val = np.load(ARRAY_ROOT / f"y_val{suffix}", mmap_mode="r")
    expected_shape = (CONFIG["num_frequency_bins"], CONFIG["sequence_length"]) if CONFIG["input_modality"] == "spectrogram" else (1, CONFIG["sequence_length"])
    assert x_train.shape == (840, *expected_shape) and x_val.shape == (210, *expected_shape)
    assert np.array_equal(y_train, [int(row['label_binary']) for row in fold_train_rows])
    assert np.array_equal(y_val, [int(row['label_binary']) for row in fold_val_rows])

monitor_x = np.load(ARRAY_ROOT / "X_val_vowels.npy", mmap_mode="r")
monitor_y = np.load(ARRAY_ROOT / "y_val_vowels.npy", mmap_mode="r")
assert monitor_x.shape == (150, *expected_shape) and monitor_y.shape == (150,)
monitor_loader = DataLoader(
    TensorDataset(torch.from_numpy(np.array(monitor_x)), torch.from_numpy(np.array(monitor_y))),
    batch_size=CONFIG['batch_size'], shuffle=False,
)
print("Cinco folds verificados: 56/14 personas por fold; val global: 10 personas.")
print("Entrada:", expected_shape, "CNN:", "2D" if CONFIG["input_modality"] == "spectrogram" else "1D")
print("Test no se ha cargado.")


## 4. Evolución y seguimiento en val

El fitness usa las cinco validaciones internas de `train`. Al cerrar cada generación, el mejor checkpoint se evalúa en el `val` global y se guardan F1 y accuracy en `generation_stats`. **Ningún array de test se carga durante esta etapa.**


In [ ]:
class MonitoredVowelsEvolution(HybridNeuroevolution):
    def evaluate_population(self):
        super().evaluate_population()
        if not self.best_checkpoint_path:
            return
        _, model = self.load_best_checkpoint()
        if model is None:
            raise RuntimeError("No se pudo cargar el checkpoint para monitorizar val")
        model.eval()
        targets, predictions = [], []
        with torch.no_grad():
            for features, labels in monitor_loader:
                logits = model(features.to(self.device))
                predictions.extend(logits.argmax(dim=1).cpu().tolist())
                targets.extend(labels.tolist())
        stats = self.generation_stats[-1]
        stats['monitor_val_f1'] = 100.0 * f1_score(targets, predictions, zero_division=0)
        stats['monitor_val_accuracy'] = 100.0 * accuracy_score(targets, predictions)
        stats['monitor_val_genome_id'] = self.best_individual['id']
        print(f"Val global, generación {self.generation}: F1={stats['monitor_val_f1']:.2f}%")
        del model

start_time = datetime.now()
neuroevolution = MonitoredVowelsEvolution(CONFIG, device)
best_genome = neuroevolution.evolve()
print("Evolución completada:", datetime.now() - start_time)
print("Arquitectura elegida:", best_genome['id'])


## 5. Visualización de la evolución

In [ ]:
plot_fitness_evolution(neuroevolution, CONFIG)

monitor_stats = [stat for stat in neuroevolution.generation_stats if 'monitor_val_f1' in stat]
if monitor_stats:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([stat['generation'] for stat in monitor_stats],
            [stat['monitor_val_f1'] for stat in monitor_stats], marker='o', label='F1 en val global')
    ax.set(xlabel='Generación', ylabel='F1 (%)', title='Evolución en val global (10 personas)')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Show detailed statistics
show_evolution_statistics(neuroevolution, CONFIG)

In [ ]:
# Analyze any failed evaluations
analyze_failed_evaluations(neuroevolution)

## 6. Best Architecture Analysis

Examine the best architecture found during evolution.

In [ ]:
print("\n" + "="*80)
print("BEST ARCHITECTURE DETAILS")
print("="*80 + "\n")

best = best_genome

print(f"ID: {best['id']}")
print(f"Fitness (F1-Score): {best['fitness']:.2f}%\n")

print("ARCHITECTURE:")
print(f"  Convolutional layers: {best['num_conv_layers']}")
print(f"  Fully connected layers: {best['num_fc_layers']}")
print(f"\nCONV LAYER DETAILS:")
normalization = best.get('normalization_type', 'batch')
activations = best.get('activations', ['relu'])
for i in range(best['num_conv_layers']):
    activation = activations[i % len(activations)]
    print(f"  Conv{i+1}: {best['filters'][i]} filters, kernel={best['kernel_sizes'][i]}, norm={normalization}, activation={activation}")

print(f"\nFC LAYER DETAILS:")
for i in range(best['num_fc_layers']):
    print(f"  FC{i+1}: {best['fc_nodes'][i]} nodes")

print(f"\nHYPERPARAMETERS:")
print(f"  Optimizer: {best['optimizer']}")
print(f"  Learning rate: {best['learning_rate']}")
print(f"  Dropout rate: {best['dropout_rate']}")
print(f"  Activations: {', '.join(sorted(set(activations)))}")

if best.get('innovation_uuid'):
    print(f"\nINNOVATION TRACKING:")
    print(f"  Innovation UUID: {best['innovation_uuid'][:16]}...")
    print(f"  Number of innovation genes: {len(best.get('innovation_genes', []))}")
    if best.get('structural_history'):
        print(f"  Structural events: {len(best['structural_history'])}")

print("\n" + "="*80)

## 7. Load Best Checkpoint (Optional)

You can reload the best model from checkpoint for inference or further analysis.

In [ ]:
# Load best checkpoint
loaded_genome, loaded_model = neuroevolution.load_best_checkpoint()

if loaded_model is not None:
    print("\nBest model loaded successfully!")
    print(f"Model has {sum(p.numel() for p in loaded_model.parameters())} parameters")
    print(f"Trainable parameters: {sum(p.numel() for p in loaded_model.parameters() if p.requires_grad)}")
    print("\nModel architecture:")
    print(loaded_model)
else:
    print("No checkpoint available to load")

## 8. Evaluación final: único test global

Esta celda se ejecuta **solo después** de `evolve()`. Entrena desde cero la arquitectura elegida usando `train.csv`, usa `val.csv` para elegir el checkpoint y abre `X_test_vowels.npy` por primera vez para calcular las métricas finales. No se repite test por fold.


In [ ]:
def global_loader(split: str, *, shuffle: bool) -> DataLoader:
    x = np.load(ARRAY_ROOT / f"X_{split}_vowels.npy", mmap_mode="r")
    y = np.load(ARRAY_ROOT / f"y_{split}_vowels.npy", mmap_mode="r")
    return DataLoader(
        TensorDataset(torch.from_numpy(np.array(x)), torch.from_numpy(np.array(y))),
        batch_size=CONFIG['batch_size'], shuffle=shuffle,
    )

final_train_loader = global_loader('train', shuffle=True)
final_val_loader = global_loader('val', shuffle=False)
# Primer acceso al conjunto test en este notebook: búsqueda y selección terminadas.
final_test_loader = global_loader('test', shuffle=False)
assert len(final_train_loader.dataset) == 1050
assert len(final_val_loader.dataset) == 150
assert len(final_test_loader.dataset) == 300

final_test_results = evaluate_single_fold(
    best_genome, CONFIG, final_train_loader, final_val_loader,
    final_test_loader, fold_num=1, device=device,
    num_epochs=CONFIG['num_epochs'],
)
final_test_results.pop('fold', None)  # One global holdout, not a fold.
final_test_results['protocol'] = 'single_global_person_held_out_test'
final_test_results['train_people'] = 70
final_test_results['validation_people'] = 10
final_test_results['test_people'] = 20
final_test_results['dataset'] = 'vowels'
final_test_results['confusion_matrix'] = np.asarray(final_test_results['confusion_matrix']).tolist()
results_path = Path(CONFIG['artifacts_dir']) / 'vowels_final_test.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
def json_numpy(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Unsupported result type: {type(value).__name__}")

with results_path.open('w', encoding='utf-8') as stream:
    json.dump(final_test_results, stream, indent=2, default=json_numpy)
print('Evaluación final guardada en:', results_path)


In [ ]:
print('RESULTADOS DEL ÚNICO TEST GLOBAL')
for metric in ('accuracy', 'sensitivity', 'specificity', 'f1_score', 'auc'):
    print(f"{metric}: {final_test_results[metric]:.2f}%")
print('Audios de test:', final_test_results['n_samples'])

matrix = np.asarray(final_test_results['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap='Blues')
for row in range(2):
    for column in range(2):
        ax.text(column, row, str(matrix[row, column]), ha='center', va='center')
ax.set(xticks=[0, 1], yticks=[0, 1], xlabel='Predicción', ylabel='Etiqueta real',
       title='Matriz de confusión: test global (20 personas)')
plt.tight_layout()
plt.show()


## Resumen del protocolo

1. Preparar los WAV y las dos modalidades `.npy` con `vowels_dataset_por_persona.ipynb`.
2. Elegir `CONFIG["input_modality"]` en `config.py`. Evolucionar con cinco folds de `train` (70 personas); monitorizar en `val` (10 personas).
3. Fijar arquitectura y reglas de entrenamiento.
4. Entrenar la arquitectura elegida y evaluar una sola vez el `test` global (20 personas).

Las métricas de `val` sirven para desarrollo. La estimación final independiente procede del único `test` global. Los resultados del modelo se guardan en `artifacts/test_vowels_audio/` o `artifacts/test_vowels_spectrogram/`, segun la modalidad.